# Build 04-04 · Regime-break robustness — does the v3 corruption shift survive real cutoffs, not the median placeholder?

**Name.** Called "falsification" in earlier discussion — renamed here because that word reads as
"is this fake", when what it actually checks is **robustness of a sign/magnitude to which cutoff
you pick**. If you prefer a different name: `regime_break_robustness` (used here),
`placebo_cutoff_check`, `policy_break_stability` are the other candidates that were on the table.

**What this checks.** 04_01 split each version's own window into early/late by the **median
date** — explicitly flagged there as a PLACEHOLDER, "revisit once real dates are visible." v2's
own serving history moved its τ four times (`config.DECISION_RULES["v2"]["regimes"]`), and two of
those breaks (2024-06-02, 2026-02-25) fall INSIDE v3's train/OOT window — because v3 trains on
`model_v2_observed_outcome`, i.e. on v2's serving log, which is where those regimes actually live.
**Those break dates are non-arbitrary cutoffs the median split ignores.** This notebook re-tags
v3 with each real break as the era cutoff instead, and checks whether `DiD(v3)` keeps the same
sign and rough size as 04_02's median-placeholder number. If it flips sign depending on which
cutoff you pick, that undermines trusting the median-split headline number; if it doesn't, that is
independent evidence the headline finding isn't an artefact of an arbitrarily-placed cutoff.

**Scope.** v2 is NOT checked here — its own train/OOT window has ZERO internal regime variation
(it predates v2's 2021-06-03 deployment entirely, see [[project-v1v2-inheritance-dead]] discussion
in chat 2026-09-12), so there is no real break to substitute for its median split; only v3 has one.

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config
import schema
import figstyle
import feature_alias
from loaders import load
from estimator import concentration

figstyle.apply()
pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — which of v2's regime breaks actually fall inside v3's window? Derived, never hand-copied
# (config.spans_a_break reads config.DECISION_RULES["v2"]["regimes"] — see 04_01 Notes).
ID_COL = schema.CLAIM_ID
TAG_COLS = [schema.DATE, "score", schema.DECISION, "region", "era", schema.OBSERVED]
V3_SPLITS = list(dict.fromkeys(["train", config.OOT_SPLIT["v3"]]))


def window_bounds(version: str, split: str) -> tuple:
    """[min date, max date) actually present in that (version, split)'s targets -- not a
    hard-coded window, so this stays correct if the export window ever changes."""
    d = load(version, split=split)
    dates = pd.to_datetime(d.frame[schema.DATE])
    return dates.min(), dates.max() + pd.Timedelta(days=1)


candidate_breaks = {}
for split in V3_SPLITS:
    start, end = window_bounds("v3", split)
    brks = config.spans_a_break("v2", start, end)
    candidate_breaks[split] = [b["date"] for b in brks]
    print(f"v3/{split}: window {start.date()} .. {end.date()}  ->  "
          f"{len(brks)} v2 regime break(s) inside it: {candidate_breaks[split]}")

In [ ]:
# §2 — re-tag v3 with a FIXED cutoff date (a real break) instead of 04_01's median, in memory
# only. Never written as a shap_did_input file -- this is a robustness check on the median split,
# not a second headline artefact.
def region_and_era_at(version: str, split: str, cutoff) -> pd.DataFrame:
    d = load(version, split=split)
    df = d.frame.copy()
    df[schema.DECISION] = d.decisions
    df["region"] = np.where(df[schema.DECISION] == 1, "B", "A")
    dates = pd.to_datetime(df[schema.DATE])
    if getattr(dates.dt, "tz", None) is not None:
        # drop the zone, keep the wall clock -- same rule as threshold.apply / ReweightCorrector.
        # _tau / 03_02's §2a (real dates are tz-aware; cutoff below is a plain ISO date/regime
        # break with no zone, so comparing them directly raises "Cannot compare tz-naive and
        # tz-aware datetime-like objects" -- config.spans_a_break in §1 never hits this because
        # it routes both sides through _as_date() (-> datetime.date, tz-blind) instead.
        dates = dates.dt.tz_localize(None)
    cutoff = pd.Timestamp(cutoff)
    df["era"] = np.where(dates <= cutoff, "early", "late")
    return df[[ID_COL, "region", "era"]]


def cell_simpson(table: pd.DataFrame, **filters) -> float:
    sub = table
    for col, val in filters.items():
        sub = sub[sub[col] == val]
    drop_cols = [c for c in TAG_COLS if c in sub.columns]
    mabs = concentration.mean_abs(sub.drop(columns=drop_cols, errors="ignore"), id_col=ID_COL)
    return concentration.simpson(mabs)


def did_at_cutoff(version: str, split: str, cutoff) -> dict:
    tags = region_and_era_at(version, split, cutoff)
    attrs = load(version, split=split).attributions
    table = tags.merge(attrs, on=ID_COL, how="inner")
    s = {(r, e): cell_simpson(table, region=r, era=e) for r in ("A", "B") for e in ("early", "late")}
    did = (s[("B", "late")] - s[("A", "late")]) - (s[("B", "early")] - s[("A", "early")])
    n_early = int((tags["era"] == "early").sum())
    n_late = int((tags["era"] == "late").sum())
    return {"cutoff": pd.Timestamp(cutoff).date(), "did": did,
            "n_early": n_early, "n_late": n_late, "n_claims": len(table)}

In [ ]:
# §3 — run it: median placeholder (04_01/04_02's own cutoff) vs each real regime break inside
# the window, for every v3 split that actually contains one
rows = []
for split in V3_SPLITS:
    meta_path = config.split_path("shap_did_input", "v3", split).with_name(
        config.split_path("shap_did_input", "v3", split).stem + "_meta.json")
    if meta_path.exists():
        median_cutoff = json.loads(meta_path.read_text(encoding="utf-8"))["era_cutoff_date"]
        row = did_at_cutoff("v3", split, median_cutoff)
        row.update(split=split, cutoff_kind="04_01 median placeholder")
        rows.append(row)
    else:
        print(f"[v3/{split}] shap_did_input meta not found -- run 04_01 first for a median baseline")

    for brk in candidate_breaks[split]:
        row = did_at_cutoff("v3", split, brk)
        row.update(split=split, cutoff_kind="real regime break")
        rows.append(row)

robustness = pd.DataFrame(rows).set_index(["split", "cutoff_kind", "cutoff"])
display(robustness)
# Index is (split, cutoff_kind, cutoff), not a feature name -- plain export, no alias twin needed.
figstyle.save_table(robustness, "03_regime_break_robustness")

signs = robustness["did"].apply(lambda x: "+" if x > 0 else ("-" if x < 0 else "0"))
if signs.nunique() <= 1:
    print("\nAll cutoffs agree in SIGN -- the median-placeholder headline is not an artefact of")
    print("which cutoff was picked.")
else:
    print("\nSIGN FLIPS across cutoffs -- do not trust the median-placeholder headline number")
    print("without reconciling why the real regime breaks disagree with it.")

## Notes

- **v2 has no analogue here, for two separate reasons, not one.** (1) Its train/OOT window
  (2018-01 to 2020-09) predates v2's own 2021-06-03 deployment entirely, so it has zero
  internal regime variation to test against. (2) More fundamentally: during that window
  decisions were made under **v1's** rule (segmented on mobility, 0.75/0.85) — not any v2
  regime at all, since v2 was not yet live — and this population was **never fed into v3's
  training** (v3 trains on v2's LIVE SERVING output, a later and different population).
  `d.decisions` for v2 on this window is a retrospective/counterfactual relabelling ("if v2's
  rule had applied here"), not a description of what actually happened, and is disconnected
  from the SFP chain regardless of regime constancy. Don't reach for it as a substitute
  population for a v2 within-version check — see [[project-v2-train-window-not-connected]].
- **This does not replace 04_01's median split** as the headline era rule — it only asks whether
  the headline conclusion (sign, rough size) is robust to swapping in the two dates that actually
  mean something. If robust, 04_01's placeholder note can be downgraded from "must fix" to "known
  simplification, checked."
- **The error-inheritance analogue of this idea is a SEPARATE, still-unbuilt notebook** —
  `notebook/real/detection/02_error_inheritance.ipynb` (moved there 2026-09-12) has its own
  "regime-flip falsification, unrun" item (does the estimated effect's SIGN flip at each of v2's 4
  breaks) described in `problem.md` but not implemented. Don't conflate the two — that one tests a
  DIFFERENT estimator (error inheritance), on possibly different splits, and stays a separate task.
- Not yet run against real data.